In [3]:
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import math
from chunking_embedding import *
from pathlib import Path

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
def get_embeddings(chunks, batch_size=8):
    return model.encode(chunks, convert_to_tensor=True, show_progress_bar=True, batch_size=batch_size)

In [ ]:
get_embeddings(["Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek"])

In [ ]:
chunks = [
    '''The story begins in Springfield, Massachusetts, 1891. Dr. James Naismith's simple invention
of nailing peach baskets to a gymnasium wall has evolved into something extraordinary.
Today's basketball landscape showcases a sport that has broken free from its American
roots, becoming a cultural force that bridges continents and cultures. This transformation
reflects not just athletic evolution, but a broader story of global connectivity and shared
passion''',
    '''The NBA's transformation tells a compelling story of basketball's globalization. Gone are the
days when the league was predominantly American. Today's NBA features transcendent
international talents like Nikola Jokić, Joel Embiid, and Giannis Antetokounmpo – players
who have redefined excellence in the sport. Their success represents more than individual
achievement; it symbolizes basketball's power to discover and nurture talent regardless of
origin''',
    '''The statistics paint a vivid picture of basketball's global reach. FIBA's latest reports indicate
that over 450 million people actively play basketball worldwide. The NBA's global broadcast
reaches 215 countries and territories in 47 languages. In the 2023-24 season, 125
international players from 40 countries graced NBA rosters on opening night. Perhaps most
strikingly, China alone boasts 300 million basketball players – a number that exceeds the
entire U.S. population.''',
    '''The women's game has written its own remarkable chapter in basketball's global story. The
WNBA continues to expand its international influence, with stars like Jonquel Jones and Ezi
Magbegor leading the charge. As WNBA Commissioner Cathy Engelbert notes, "Women's
basketball is experiencing unprecedented growth." This growth manifests in increased
viewership, engagement, and participation across demographics, creating new role models
for aspiring female athletes worldwide'''
]

In [16]:
user_input = 'Where does the story begin?'
question_embedding = get_embeddings(chunks=user_input)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.78s/it]


In [21]:
def cosine_sim(a, b):
    if len(a) != len(b):
        raise ValueError("Embedding lengths do not match")
    
    dot_value = 0
    magnitude_a = 0
    magnitude_b = 0

    for i, j in zip(a, b):
        dot_value += i * j
        magnitude_a += i**2
        magnitude_b += j**2
    
    return dot_value / (magnitude_a**0.5 * magnitude_b**0.5)

In [22]:
cosine_sim(a=[1, 2, 3], b=[4, 5, 6])

0.9746318461970762

In [13]:
PROMPT = '''
You are a helpful teaching assistant. Your task is to answer the user's question given the context of the information. 
Always use the information given to you to answer the question, and do not make anything up. 

Question: {}

Context: {}
'''

In [18]:
USER_PROMPT = PROMPT.format(user_input, relevent_chunk)

NameError: name 'relevent_chunk' is not defined

In [19]:
print(USER_PROMPT)

NameError: name 'USER_PROMPT' is not defined

In [2]:
def extract_text(pdf_bytes: bytes) -> str:
    """
    Extract all text from a PDF using pymupdf
    """
    doc = pymupdf.open(stream=pdf_bytes, filetype='pdf')
    return "\n\n".join(page.get_text("text") for page in doc)

In [8]:
test_path = Path("../data/raw/test pdfs/basketball_pdf.pdf").resolve()

In [9]:
extract_text(test_path.read_bytes())

'The Global Game: Basketball\'s Unifying\nPower in the Modern Era\nFrom Peach Baskets to Global Phenomenon\nThe New Face of Professional Basketball\nNumbers That Speak Volumes\nWomen Breaking Barriers\nThe Economic Slam Dunk\nTechnology: The Game Changer\nYouth Development: Planting Seeds for Tomorrow\nUrban Renaissance Through Basketball\nThe Future Game\nConclusion: More Than Just a Game\n1\n\n\nFrom Peach Baskets to Global Phenomenon\nThe story begins in Springfield, Massachusetts, 1891. Dr. James Naismith\'s simple invention\nof nailing peach baskets to a gymnasium wall has evolved into something extraordinary.\nToday\'s basketball landscape showcases a sport that has broken free from its American\nroots, becoming a cultural force that bridges continents and cultures. This transformation\nreflects not just athletic evolution, but a broader story of global connectivity and shared\npassion.\nThe New Face of Professional Basketball\nThe NBA\'s transformation tells a compelling story o

In [32]:
embedding_sim_score: dict = {c: cosine_sim(ce, question_embedding).item() for c, ce in enumerate(pdf_embeddings)}
relevant_chunk_ids = (sorted(embedding_sim_score, key=embedding_sim_score.get, reverse=True)[:3])
relevant_chunks = [chunks[id] for id in relevant_chunk_ids]

In [38]:
for i in relevant_chunks:
    print(i)
    print()

From Peach Baskets to Global Phenomenon
The story begins in Springfield, Massachusetts, 1891. Dr. James Naismith's simple invention
of nailing peach baskets to a gymnasium wall has evolved into something extraordinary.
Today's basketball landscape showcases a sport that has broken free from its American
roots, becoming a cultural force that bridges continents and cultures. This transformation
reflects not just athletic evolution, but a broader story of global connectivity and shared
passion.
The New Face of Professional Basketball
The NBA's transformation tells a compelling story of basketball's globalization. Gone are the
days when the league was predominantly American. Today's NBA features transcendent
international talents like Nikola Jokić, Joel Embiid, and Giannis Antetokounmpo – players
who have redefined excellence in the sport.

The rise of 3x3 basketball as an Olympic sport
opens new competitive avenues for nations with limited resources, while technological
innovations promis

In [17]:
chunks = chunk_text(extract_text(test_path.read_bytes()), goal="semantic", chunk_size=1000, overlap=100)
pdf_embeddings = get_embeddings(chunks=chunks)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.32it/s]


In [54]:
def format_relevant_chunks(relevant_chunks):
    formatted_str = ""

    for idx, chunk in enumerate(relevant_chunks):
        formatted_str += f"CHUNK #{idx+1}: \n{chunk} \n\n"
    
    return formatted_str

In [55]:
print(format_relevant_chunks(relevant_chunks=relevant_chunks))

CHUNK #1: 
From Peach Baskets to Global Phenomenon
The story begins in Springfield, Massachusetts, 1891. Dr. James Naismith's simple invention
of nailing peach baskets to a gymnasium wall has evolved into something extraordinary.
Today's basketball landscape showcases a sport that has broken free from its American
roots, becoming a cultural force that bridges continents and cultures. This transformation
reflects not just athletic evolution, but a broader story of global connectivity and shared
passion.
The New Face of Professional Basketball
The NBA's transformation tells a compelling story of basketball's globalization. Gone are the
days when the league was predominantly American. Today's NBA features transcendent
international talents like Nikola Jokić, Joel Embiid, and Giannis Antetokounmpo – players
who have redefined excellence in the sport. 

CHUNK #2: 
The rise of 3x3 basketball as an Olympic sport
opens new competitive avenues for nations with limited resources, while technolog